In [6]:
import pandas as pd

placeholders = ["", "N/A", "unknown", "-"]

for column in clean_df.select_dtypes(include="object").columns:
    clean_df[column] = clean_df[column].replace(
        placeholders, pd.NA
    )

#Working on blank values
clean_df = df.copy()

#Check missing values again:
print(clean_df.isna().sum())

#Calculate percentages:
missing_percentage = (
    clean_df.isna().mean() * 100
).round(2)

print(missing_percentage)



#Check exact duplicate rows
print(clean_df.duplicated().sum())

#Display all rows in duplicate groups:
duplicates = clean_df[
    clean_df.duplicated(keep=False)
]
print(duplicates)

#Check repeated transaction-line IDs
repeated_ids = clean_df[
    clean_df.duplicated(
        subset=["Transaction_Line_ID"],
        keep=False
    )
]
print(repeated_ids)

#Remove confirmed exact duplicates
clean_df = clean_df.drop_duplicates()
#For a confirmed duplicate transaction-line ID, you could use:
clean_df = clean_df.drop_duplicates(
    subset=["Transaction_Line_ID"],
    keep="first"
)

#Check the result
print("Remaining exact duplicates:",
      clean_df.duplicated().sum())
print("Remaining repeated transaction IDs:",
      clean_df["Transaction_Line_ID"].duplicated().sum())



#Inspect distinct category values
print(clean_df["Category"].value_counts(dropna=False))

#Remove unnecessary spaces
clean_df["Category"] = (
    clean_df["Category"]
    .astype("string")
    .str.strip()
)

#Standardise capitalisation
clean_df["Category"] = (
    clean_df["Category"]
    .str.title()
)

#Use a mapping dictionary
category_mapping = {
    "confirmed_old_label_1": "Correct Label 1",
    "confirmed_old_label_2": "Correct Label 2"
}

clean_df["Category"] = clean_df["Category"].replace(
    category_mapping
)

#Verify the results
print(clean_df["Category"].value_counts(dropna=False))

#ensure that equivalent categories have been consolidated without combining genuinely different categories.
print(clean_df["Category"].unique())

#remove the text units and any unnecessary spaces
quantity_text = (
    clean_df["Quantity"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s*units?$", "", regex=True)
)

#converts valid numeric strings into numbers and turns unparseable values into missing values
clean_df["Quantity"] = pd.to_numeric(
    quantity_text,
    errors="coerce"
)

#Remove currency symbols and thousands separators
def clean_numeric(series):
    text = series.astype("string").str.strip()

    text = text.str.replace(",", "", regex=False)
    text = text.str.replace("₹", "", regex=False)

    return pd.to_numeric(text, errors="coerce")

clean_df["Unit_Price"] = clean_numeric(
    clean_df["Unit_Price"]
)

clean_df["Unit_Cost"] = clean_numeric(
    clean_df["Unit_Cost"]
)

#Convert Discount
def clean_discount(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text.endswith("%"):
        number = float(text[:-1].replace(",", ""))
        return number / 100

    return float(text)

clean_df["Discount"] = clean_df["Discount"].apply(
    clean_discount
)

#Convert sells and profit
clean_df["Sales"] = clean_numeric(clean_df["Sales"])

clean_df["Profit"] = clean_numeric(clean_df["Profit"])

#verify sells conversion
print(clean_df.dtypes)

#for compact check
print(clean_df[
    ["Order_Date", "Quantity", "Unit_Price",
     "Unit_Cost", "Discount", "Sales", "Profit"]
].dtypes)

#Check whether conversion introduced new missing values
print(clean_df.isna().sum())

#Total revenue
total_revenue = clean_df["Sales"].sum()

print("Total Revenue:", total_revenue)

#Calculate revenue by Region
revenue_by_region = (
    clean_df.groupby("Region")["Sales"]
    .sum()
)

print(revenue_by_region)

#Calculate Tottal orders
total_orders = clean_df["Order_ID"].nunique()
print("Total Orders:", total_orders)
#frequency of oredr
order_frequency = clean_df["Order_ID"].value_counts()
print(order_frequency.head())

#Average oreder value AOV
total_revenue = clean_df["Sales"].sum()
total_orders = clean_df["Order_ID"].nunique()
average_order_value = total_revenue / total_orders
print("Average Order Value:", average_order_value)

#For order-level comparisons, first aggregate line revenue by Order_ID
order_revenue = (
    clean_df.groupby("Order_ID")["Sales"]
    .sum()
)
print(order_revenue.head())

aov = order_revenue.mean()
print("Average Order Value:", aov)

#Total unit sold
total_units = clean_df["Quantity"].sum()
print("Total Units Sold:", total_units)
#Find unit sold by considering product
units_by_product = (
    clean_df.groupby(
        ["Product_ID", "Product_Name"]
    )["Quantity"]
    .sum()
    .sort_values(ascending=False)
)
print(units_by_product)
#Unit sold by category
units_by_category = (
    clean_df.groupby("Category")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)
print(units_by_category)

#Revenue By product
revenue_by_product = (
    clean_df.groupby(
        ["Product_ID", "Product_Name"]
    )["Sales"]
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_product)
#Revenue by category
revenue_by_category = (
    clean_df.groupby("Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_category)
#Print Top 5 productsa
top_5_products = revenue_by_product.head(5)
print(top_5_products)
#visualise result
import matplotlib.pyplot as plt
top_5_products.plot(kind="bar")
plt.title("Top 5 Products by Revenue")
plt.xlabel("Product")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'clean_df' is not defined